# 05 — Facilities Metric Silver

Builds `public_health.silver.facilities_metric_silver` — unified metric table from all sources described in facilities-silver-plan v1.4. Combines NHFR, HFSRB, PSA, and FHSIS metrics for regional/national/province/city hospital and primary care facility capacity, beds, and coverage.

### Sources (bronze)
- PSA: hospitals and beds (region/national/legacy)
- FHSIS: primary care facility counts
- NHFR/HFSRB: hospital registry alignment for facility keys, coverage mapping

### Geography mapping
Resolved via `public_health.silver.geography_silver`. Maps national, region, province, city/municipality, barangay codes and names.

In [0]:
# Imports and config
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import uuid

PSA_REGION_ANNUAL = "public_health.bronze.facilities_psa_hospitals_region_annual_bronze"
PSA_BEDS_REGION_ANNUAL = "public_health.bronze.facilities_psa_beds_region_annual_bronze"
PSA_HOSP_NAT_ANNUAL = "public_health.bronze.facilities_psa_hospitals_national_annual_bronze"
PSA_HOSP_BEDS_NAT_ANNUAL = "public_health.bronze.facilities_psa_hospitals_beds_national_annual_bronze"
PSA_HOSP_REGION_LEGACY = "public_health.bronze.facilities_psa_hospitals_region_annual_legacy_bronze"
PSA_BHS_REGION_ANNUAL = "public_health.bronze.facilities_psa_bhs_region_annual_bronze"
FHSIS_PRIMARY_CARE = "public_health.bronze.facilities_fhsis_primary_care_bronze"
METADATA_TABLE = "public_health.bronze.ingestion_metadata"
GEOGRAPHY_SILVER = "public_health.silver.geography_silver"
TARGET_TABLE = "public_health.silver.facilities_metric_silver"
SILVER_BUILD_ID = str(uuid.uuid4())

def clean_numeric(col_expr):
    """Clean string numeric: remove commas, handle non-numeric tokens (*, #DIV/0!, -, etc)."""
    s = F.trim(col_expr.cast("string"))
    cleaned = F.regexp_replace(s, ",", "")
    return F.when(cleaned.rlike(r"^-?\d+(\.\d+)?$"), cleaned.cast("double")).otherwise(F.lit(None))

print(f"Silver build ID: {SILVER_BUILD_ID}")
print(f"Target table: {TARGET_TABLE}")
print("Helper: clean_numeric() defined")

In [0]:
# ============================================================
# Build area_name -> (area_key, area_level) mapping
# Covers all source naming conventions for regions + national
# Province/municipality names resolved from geography_silver
# ============================================================

region_map = [
    # (raw_area_name, area_key, area_level)
    # PSA region annual names
    ("Total Number of Hospitals", "PH", "national"),
    ("NCR - National Capital Region", "1300000000", "region"),
    ("Cordillera Administrative Region", "1400000000", "region"),
    ("Region I - Ilocos Region", "0100000000", "region"),
    ("Region II - Cagayan Valley", "0200000000", "region"),
    ("Region III - Central Luzon", "0300000000", "region"),
    ("Region IV-A  CALABARZON", "0400000000", "region"),
    ("Region IV-A CALABARZON", "0400000000", "region"),
    ("MIMAROPA Region", "1700000000", "region"),
    ("Region V - Bicol Region", "0500000000", "region"),
    ("Region VI - Western Visayas", "0600000000", "region"),
    ("Region VII - Central Visayas", "0700000000", "region"),
    ("Region VIII - Eastern Visayas", "0800000000", "region"),
    ("Region IX - Zamboanga Peninsula", "0900000000", "region"),
    ("Region X - Northern Mindanao", "1000000000", "region"),
    ("Region XI - Davao Region", "1100000000", "region"),
    ("Region XII - SOCCSKSARGEN", "1200000000", "region"),
    ("Region XIII - Caraga", "1600000000", "region"),
    ("BARMM", "1900000000", "region"),
    # BHS column names
    ("philippines", "PH", "national"),
    ("ncr", "1300000000", "region"),
    ("car", "1400000000", "region"),
    ("region_01", "0100000000", "region"),
    ("region_02", "0200000000", "region"),
    ("region_03", "0300000000", "region"),
    ("region_04a", "0400000000", "region"),
    ("mimaropa", "1700000000", "region"),
    ("region_05", "0500000000", "region"),
    ("region_06", "0600000000", "region"),
    ("region_07", "0700000000", "region"),
    ("region_08", "0800000000", "region"),
    ("region_09", "0900000000", "region"),
    ("region_10", "1000000000", "region"),
    ("region_11", "1100000000", "region"),
    ("region_12", "1200000000", "region"),
    ("caraga", "1600000000", "region"),
    ("barmm", "1900000000", "region"),
    # Legacy region codes
    ("NCR", "1300000000", "region"),
    ("CAR", "1400000000", "region"),
    ("I", "0100000000", "region"),
    ("II", "0200000000", "region"),
    ("III", "0300000000", "region"),
    ("IV-A", "0400000000", "region"),
    ("MIMAROPA", "1700000000", "region"),
    ("V", "0500000000", "region"),
    ("VI", "0600000000", "region"),
    ("VII", "0700000000", "region"),
    ("VIII", "0800000000", "region"),
    ("IX", "0900000000", "region"),
    ("X", "1000000000", "region"),
    ("XI", "1100000000", "region"),
    ("XII", "1200000000", "region"),
    ("XIII", "1600000000", "region"),
    ("ARMM", "1900000000", "region"),
    # FHSIS names
    ("Philippines", "PH", "national"),
    ("PHILIPPINES", "PH", "national"),
    ("Region 1", "0100000000", "region"),
    ("Region 2", "0200000000", "region"),
    ("Region 3", "0300000000", "region"),
    ("Region 4A", "0400000000", "region"),
    ("Region 4B", "1700000000", "region"),
    ("Region 5", "0500000000", "region"),
    ("Region 6", "0600000000", "region"),
    ("Region 7", "0700000000", "region"),
    ("Region 8", "0800000000", "region"),
    ("Region 9", "0900000000", "region"),
    ("Region 10", "1000000000", "region"),
    ("Region 11", "1100000000", "region"),
    ("Region 12", "1200000000", "region"),
    ("Region 13", "1600000000", "region"),
]

# Build hardcoded mapping DataFrame
region_map_df = spark.createDataFrame(region_map, ["area_name", "area_key", "area_level"])

# Build province + municipality mapping from geography_silver
geo = spark.table(GEOGRAPHY_SILVER)
province_map = (
    geo.filter(F.col("geographic_level") == "Prov")
    .select(
        F.trim(F.col("area_name")).alias("area_name"),
        F.col("psgc_code").alias("area_key"),
        F.lit("province").alias("area_level"),
    )
)
muni_map = (
    geo.filter(F.col("geographic_level").isin("Mun", "City"))
    .select(
        F.trim(F.col("area_name")).alias("area_name"),
        F.col("psgc_code").alias("area_key"),
        F.lit("municipality").alias("area_level"),
    )
)

# Union: hardcoded region mappings take priority over geo-silver names
all_mappings = region_map_df.unionByName(province_map).unionByName(muni_map)

# Deduplicate by area_name -- keep hardcoded (region/national) entries first
w = Window.partitionBy("area_name").orderBy(
    F.when(F.col("area_level") == "national", 0)
    .when(F.col("area_level") == "region", 1)
    .otherwise(2)
)
area_lookup = (
    all_mappings
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f"Area lookup entries: {area_lookup.count()}")
area_lookup.groupBy("area_level").count().show()

In [0]:
# ============================================================
# Normalize all 6 PSA sources to long format
# Each row: (area_name, year, ownership, metric_name, metric_value,
#   source_name, source_priority, _source_key, _source_file, _source_sheet,
#   _source_row_num, _as_of_date, _ingested_at)
# ============================================================
from functools import reduce

LINEAGE_COLS = ["_source_key", "_source_file", "_source_sheet", "_source_row_num", "_as_of_date", "_ingested_at"]

# --- 1. PSA hospitals region annual (wide: Region + Total/Gov/Pvt x 2020-2024) ---
psa_hosp_reg = spark.table(PSA_REGION_ANNUAL)
years = [2020, 2021, 2022, 2023, 2024]
own_map = [("Total", "total"), ("Government", "government"), ("Private", "private")]

parts = []
for yr in years:
    for prefix, own in own_map:
        parts.append(
            psa_hosp_reg.select(
                F.col("Region").alias("area_name"),
                F.lit(yr).alias("year"),
                F.lit(own).alias("ownership"),
                F.lit("hospitals_count").alias("metric_name"),
                clean_numeric(F.col(f"{prefix}_{yr}")).alias("metric_value"),
                F.lit("psa_hospitals_region_annual").alias("source_name"),
                F.lit(10).alias("source_priority"),
                *LINEAGE_COLS,
            )
        )
hosp_reg_long = reduce(lambda a, b: a.unionByName(b), parts)

# --- 2. PSA beds region annual (same wide structure, column = Regions) ---
psa_beds_reg = spark.table(PSA_BEDS_REGION_ANNUAL)
parts = []
for yr in years:
    for prefix, own in own_map:
        parts.append(
            psa_beds_reg.select(
                F.col("Regions").alias("area_name"),
                F.lit(yr).alias("year"),
                F.lit(own).alias("ownership"),
                F.lit("beds_count").alias("metric_name"),
                clean_numeric(F.col(f"{prefix}_{yr}")).alias("metric_value"),
                F.lit("psa_beds_region_annual").alias("source_name"),
                F.lit(10).alias("source_priority"),
                *LINEAGE_COLS,
            )
        )
beds_reg_long = reduce(lambda a, b: a.unionByName(b), parts)

# --- 3. PSA hospitals national annual (long, 2006-2015) ---
psa_nat = spark.table(PSA_HOSP_NAT_ANNUAL)
parts = []
for col_name, own in [("Total", "total"), ("Government", "government"), ("Private", "private")]:
    parts.append(
        psa_nat.select(
            F.lit("Philippines").alias("area_name"),
            F.col("Year").cast("int").alias("year"),
            F.lit(own).alias("ownership"),
            F.lit("hospitals_count").alias("metric_name"),
            clean_numeric(F.col(col_name)).alias("metric_value"),
            F.lit("psa_hospitals_national_annual").alias("source_name"),
            F.lit(8).alias("source_priority"),
            *LINEAGE_COLS,
        )
    )
nat_long = reduce(lambda a, b: a.unionByName(b), parts)

# --- 4. PSA hospitals beds national annual (long, 1990-2015) ---
psa_beds_nat = spark.table(PSA_HOSP_BEDS_NAT_ANNUAL).filter(F.col("year").rlike("^\\d{4}$"))
parts = []
for metric_col, metric_name, own in [
    ("hospitals_total", "hospitals_count", "total"),
    ("hospitals_government", "hospitals_count", "government"),
    ("hospitals_private", "hospitals_count", "private"),
    ("beds_total", "beds_count", "total"),
    ("beds_government", "beds_count", "government"),
    ("beds_private", "beds_count", "private"),
    ("beds_per_10000", "beds_per_10000", "total"),
]:
    parts.append(
        psa_beds_nat.select(
            F.lit("Philippines").alias("area_name"),
            F.col("year").cast("int").alias("year"),
            F.lit(own).alias("ownership"),
            F.lit(metric_name).alias("metric_name"),
            clean_numeric(F.col(metric_col)).alias("metric_value"),
            F.lit("psa_hospitals_beds_national_annual").alias("source_name"),
            F.lit(7).alias("source_priority"),
            *LINEAGE_COLS,
        )
    )
beds_nat_long = reduce(lambda a, b: a.unionByName(b), parts)

# --- 5. PSA BHS region annual (wide: year + region columns) ---
psa_bhs = spark.table(PSA_BHS_REGION_ANNUAL).filter(F.col("year").rlike("^\\d{4}$"))
bhs_region_cols = [
    "philippines", "ncr", "car", "region_01", "region_02", "region_03",
    "region_04a", "mimaropa", "region_05", "region_06", "region_07",
    "region_08", "region_09", "region_10", "region_11", "region_12",
    "caraga", "barmm",
]
parts = []
for col_name in bhs_region_cols:
    parts.append(
        psa_bhs.select(
            F.lit(col_name).alias("area_name"),
            F.col("year").cast("int").alias("year"),
            F.lit("total").alias("ownership"),
            F.lit("bhs_count").alias("metric_name"),
            clean_numeric(F.col(col_name)).alias("metric_value"),
            F.lit("psa_bhs_region_annual").alias("source_name"),
            F.lit(9).alias("source_priority"),
            *LINEAGE_COLS,
        )
    )
bhs_long = reduce(lambda a, b: a.unionByName(b), parts)

# --- 6. PSA hospitals region annual legacy (complex layout) ---
# Structure: col_0=Year/Total/Gov/Pvt, col_1=Philippines, col_2=NULL, col_3-col_19=regions
legacy = spark.table(PSA_HOSP_REGION_LEGACY)

# Region columns mapping: col_3=NCR, col_4=CAR, col_5=I, ... col_19=ARMM
legacy_region_cols = [
    ("col_3", "NCR"), ("col_4", "CAR"), ("col_5", "I"), ("col_6", "II"),
    ("col_7", "III"), ("col_8", "IV-A"), ("col_9", "MIMAROPA"), ("col_10", "V"),
    ("col_11", "VI"), ("col_12", "VII"), ("col_13", "VIII"), ("col_14", "IX"),
    ("col_15", "X"), ("col_16", "XI"), ("col_17", "XII"), ("col_18", "XIII"),
    ("col_19", "ARMM"),
]

# Forward-fill year from col_0 (year rows have year in col_0, data rows have Total/Gov/Pvt)
legacy = legacy.withColumn(
    "year_raw",
    F.when(F.col("col_0").rlike("^\\d{4}"), F.regexp_extract(F.col("col_0"), "\\d{4}", 0))
    .otherwise(None),
)
year_window = Window.orderBy("_source_row_num")
legacy = legacy.withColumn("year_ff", F.last("year_raw", ignorenulls=True).over(year_window))

# Keep only Total/Government/Private rows
legacy_data = legacy.filter(
    F.trim(F.col("col_0")).isin("Total", "Government", "Private")
).withColumn(
    "ownership",
    F.when(F.trim(F.col("col_0")) == "Total", "total")
    .when(F.trim(F.col("col_0")) == "Government", "government")
    .when(F.trim(F.col("col_0")) == "Private", "private"),
)

# Unpivot: col_1 (Philippines) + col_3-col_19 (regions)
parts = []
# National (col_1)
parts.append(
    legacy_data.select(
        F.lit("Philippines").alias("area_name"),
        F.col("year_ff").cast("int").alias("year"),
        F.col("ownership"),
        F.lit("hospitals_count").alias("metric_name"),
        clean_numeric(F.col("col_1")).alias("metric_value"),
        F.lit("psa_hospitals_region_annual_legacy").alias("source_name"),
        F.lit(6).alias("source_priority"),
        *LINEAGE_COLS,
    )
)
# Regions (col_3-col_19)
for col_name, region_name in legacy_region_cols:
    parts.append(
        legacy_data.select(
            F.lit(region_name).alias("area_name"),
            F.col("year_ff").cast("int").alias("year"),
            F.col("ownership"),
            F.lit("hospitals_count").alias("metric_name"),
            clean_numeric(F.col(col_name)).alias("metric_value"),
            F.lit("psa_hospitals_region_annual_legacy").alias("source_name"),
            F.lit(6).alias("source_priority"),
            *LINEAGE_COLS,
        )
    )
legacy_long = reduce(lambda a, b: a.unionByName(b), parts)

# Union all PSA sources
psa_all = hosp_reg_long.unionByName(beds_reg_long).unionByName(nat_long) \
    .unionByName(beds_nat_long).unionByName(bhs_long).unionByName(legacy_long)

print(f"PSA hospitals region annual: {hosp_reg_long.count()} rows")
print(f"PSA beds region annual: {beds_reg_long.count()} rows")
print(f"PSA national annual: {nat_long.count()} rows")
print(f"PSA beds national annual: {beds_nat_long.count()} rows")
print(f"PSA BHS region annual: {bhs_long.count()} rows")
print(f"PSA legacy: {legacy_long.count()} rows")
print(f"Total PSA rows: {psa_all.count()}")

In [0]:
# ============================================================
# Normalize FHSIS primary care and union with PSA sources
# FHSIS: province/municipality-level, year from _source_file
# Metrics: population, barangays_no, bhs_no, mhc_no, chc_no, rhu_no, health_centers_total
# No ownership breakdown (all = 'total')
# ============================================================

fhsis = spark.table(FHSIS_PRIMARY_CARE)

# Extract year from _source_file (e.g., ...province_annual_2021.xlsx -> 2021)
fhsis = fhsis.withColumn(
    "year",
    F.regexp_extract(F.col("_source_file"), r"(\d{4})\.xlsx", 1).cast("int")
)

# Define metric columns to extract
fhsis_metrics = [
    ("population", "population"),
    ("barangays_no", "barangays_no"),
    ("bhs_no", "bhs_no"),
    ("mhc_no", "mhc_no"),
    ("chc_no", "chc_no"),
    ("rhu_no", "rhu_no"),
    ("health_centers_total", "health_centers_total"),
]

parts = []
for col_name, metric_name in fhsis_metrics:
    parts.append(
        fhsis.select(
            F.trim(F.col("area_name")).alias("area_name"),
            F.col("year"),
            F.lit("total").alias("ownership"),
            F.lit(metric_name).alias("metric_name"),
            clean_numeric(F.col(col_name)).alias("metric_value"),
            F.lit("doh_fhsis_demographic").alias("source_name"),
            F.lit(5).alias("source_priority"),
            *LINEAGE_COLS,
        )
    )
fhsis_long = reduce(lambda a, b: a.unionByName(b), parts)

# Union FHSIS with PSA
all_long = psa_all.unionByName(fhsis_long)

print(f"FHSIS rows: {fhsis_long.count()}")
print(f"Total combined rows (PSA + FHSIS): {all_long.count()}")
print("\nSource breakdown:")
all_long.groupBy("source_name", "source_priority").count().orderBy(F.desc("source_priority")).show(truncate=False)
print("\nMetric breakdown:")
all_long.groupBy("metric_name").count().show(truncate=False)
print("\nYear range:")
all_long.select(F.min("year").alias("min_year"), F.max("year").alias("max_year")).show()

In [0]:
# ============================================================
# Resolve area_key via area_lookup, add lineage from ingestion_metadata,
# apply source_priority to pick winning source per metric, and write table
# ============================================================

# Join with area_lookup to resolve area_key and area_level
resolved = (
    all_long.alias("src")
    .join(
        area_lookup.alias("lkp").select("area_name", "area_key", "area_level"),
        F.trim(F.col("src.area_name")) == F.col("lkp.area_name"),
        how="left"
    )
    .drop(F.col("lkp.area_name"))
)

# Check unresolved area_names
unresolved = resolved.filter(F.col("area_key").isNull())
unresolved_count = unresolved.count()
print(f"Unresolved area_names: {unresolved_count}")
if unresolved_count > 0:
    print("Unresolved area_name samples:")
    unresolved.select("area_name", "source_name").distinct().show(30, truncate=False)

# Filter out null metric_values (empty data points) and unresolved area_keys
resolved = resolved.filter(F.col("metric_value").isNotNull() & F.col("area_key").isNotNull())

# Apply source_priority: for each (area_key, year, ownership, metric_name),
# keep the row with the highest source_priority
priority_window = Window.partitionBy("area_key", "year", "ownership", "metric_name").orderBy(F.desc("source_priority"))
resolved = (
    resolved
    .withColumn("_priority_rank", F.row_number().over(priority_window))
    .filter(F.col("_priority_rank") == 1)
    .drop("_priority_rank")
)

print(f"Rows after dedup by source_priority: {resolved.count()}")

# --- Add lineage from ingestion_metadata (deduplicated to latest per source_name) ---
metric_source_names = [
    "psa_hospitals_region_annual", "psa_beds_region_annual",
    "psa_hospitals_national_annual", "psa_hospitals_beds_national_annual",
    "psa_hospitals_region_annual_legacy", "psa_bhs_region_annual",
    "doh_fhsis_demographic",
]
meta = spark.table(METADATA_TABLE).filter(F.col("source_name").isin(metric_source_names))

# Deduplicate: take latest retrieved_at per (source_name, source_sheet)
meta_w = Window.partitionBy("source_name", "source_sheet").orderBy(F.desc("retrieved_at"))
meta_latest = (
    meta
    .withColumn("_meta_rank", F.row_number().over(meta_w))
    .filter(F.col("_meta_rank") == 1)
    .drop("_meta_rank")
    .select(
        F.col("source_name").alias("meta_source_name"),
        F.col("source_sheet").alias("meta_source_sheet"),
        F.col("batch_id").alias("_load_batch_id"),
        F.col("content_checksum").alias("_content_checksum"),
    )
)

# Join lineage
resolved = (
    resolved
    .join(
        meta_latest,
        (F.col("source_name") == F.col("meta_source_name"))
        & (F.coalesce(F.col("_source_sheet"), F.lit("")) == F.coalesce(F.col("meta_source_sheet"), F.lit(""))),
        how="left"
    )
    .drop("meta_source_name", "meta_source_sheet")
)

null_lineage = resolved.filter(F.col("_load_batch_id").isNull()).count()
print(f"Rows with missing lineage: {null_lineage}")

# --- Select final columns and write ---
final_df = resolved.select(
    F.col("area_key"),
    F.trim(F.col("area_name")).alias("area_name"),
    F.col("area_level"),
    F.col("year").cast("int"),
    F.col("ownership"),
    F.col("metric_name"),
    F.col("metric_value"),
    F.col("source_name"),
    F.col("source_priority"),
    F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
    F.col("_source_key"),
    F.col("_source_file"),
    F.col("_source_sheet"),
    F.col("_source_row_num"),
    F.col("_as_of_date"),
    F.col("_ingested_at"),
    F.col("_load_batch_id"),
    F.col("_content_checksum"),
    F.current_timestamp().alias("_silver_loaded_at"),
)

print(f"Final row count: {final_df.count()}")
print(f"Columns: {len(final_df.columns)}")

# Write: Rebuild (CREATE OR REPLACE)
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
print(f"Table written: {TARGET_TABLE}")

In [0]:
# ============================================================
# Validation
# ============================================================
result = spark.table(TARGET_TABLE)
total = result.count()
print(f"=== Validation: {TARGET_TABLE} ===")
print(f"Total rows: {total}")

# Key uniqueness: (area_key, year, ownership, metric_name) should be unique after source_priority dedup
key_cols = ["area_key", "year", "ownership", "metric_name"]
dup_keys = result.groupBy(*key_cols).agg(F.count("*").alias("cnt")).filter(F.col("cnt") > 1).count()
print(f"Duplicate keys: {dup_keys} (expected 0)")

# Null area_keys
null_keys = result.filter(F.col("area_key").isNull()).count()
print(f"Null area_keys: {null_keys} (expected 0)")

# Area level breakdown
print("\nArea level breakdown:")
result.groupBy("area_level").count().show()

# Source breakdown
print("Source breakdown:")
result.groupBy("source_name", "source_priority").count().orderBy(F.desc("source_priority")).show(truncate=False)

# Metric breakdown
print("Metric breakdown:")
result.groupBy("metric_name").count().show(truncate=False)

# Year range
print("Year range:")
result.select(F.min("year").alias("min_year"), F.max("year").alias("max_year")).show()

# Ownership breakdown
print("Ownership breakdown:")
result.groupBy("ownership").count().show()

# Sample data - hospitals_count by region, 2024
print("Sample: hospitals_count by region, 2024 (total ownership):")
result.filter((F.col("metric_name") == "hospitals_count") & (F.col("year") == 2024) & (F.col("ownership") == "total") & (F.col("area_level") == "region")) \
    .select("area_name", "metric_value", "source_name").orderBy(F.desc("metric_value")).show(20, truncate=False)

# Lineage completeness
lineage_cols = ["_silver_build_id", "_load_batch_id", "_content_checksum", "_silver_loaded_at"]
for c in lineage_cols:
    null_ct = result.filter(F.col(c).isNull()).count()
    print(f"  {c}: {null_ct} nulls")

assert dup_keys == 0, f"{dup_keys} duplicate keys"
assert null_keys == 0, f"{null_keys} null area_keys"
print("\n✅ All validation checks passed")